In [38]:
import openai, json
# url을 통해 API를 얻어오기 위해 추가
from urllib.request import urlopen, Request

client = openai.OpenAI()
messages = []
watched_movies = []

In [39]:
# 지역 날씨 가져오기
def get_weather(city):
    return "33 degree"

# 인기 영화 리스트 가져오기
def get_popular_movies():
    url = "https://nomad-movies-2.nomadcoders.workers.dev/movies"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)
    
# 영화 세부 정보 가져오기
# 위 함수에서 id만 받아서 넣어주면 되는 구조
def get_movie_details(id):
    # {}에 id를 저렇게 넣어도 된다는 게 굉장히 신기하다
    url = f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)


def get_movie_credits(id):
    # {}에 id를 저렇게 넣어도 된다는 게 굉장히 신기하다
    url = f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}/credits"

    # 이 부분은 GPT에게 문의
    # 추후 웹스크랩퍼 강의를 들어야할 것 같다
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
        },
    )
    with urlopen(request) as response:
        body = response.read().decode("utf-8")
        return json.loads(body)
    
def save_user_watched_movies(movie_titles):
    added_movies = []

    for title in movie_titles:
        title = title.strip()

        if title and title not in watched_movies:
            watched_movies.append(title)
            added_movies.append(title)

    print(f"추가된 시청 영화 목록: {added_movies}")
    print(f"총 시청 영화 목록: {watched_movies}")

    return {
        "status": "saved",
        "watched_movies": watched_movies
    }

FUNCTION_MAP = {
    "get_weather": get_weather,
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "save_user_watched_movies": save_user_watched_movies,
}

In [40]:
TOOLS = [
    {
        "type": "function",
        "function":{
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters":{
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_popular_movies",
            "description": "A function to get the popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_movie_credits",
            "description": "A function to get the movie credits by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id":{
                        "type": "integer",
                        "description": "The id of the movie to get the credits of movie.",
                    }
                },
                "required": ["id"],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name": "get_movie_details",
            "description": "A function to get the movie detail by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id":{
                        "type": "integer",
                        "description": "The id of the movie to get the detail of movie.",
                    }
                },
                "required": ["id"],
            }
        }
    },
    # 사용자가 본 영화를 저장해서 중복되지 않게 하자
    {
        "type": "function",
        "function":{
            "name": "save_user_watched_movies",
            "description": "A function to save the movies that user already watched.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_titles":{
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A List of movie that user already watched.",
                    }
                },
                "required": ["movie_titles"],
            }
        }
    },
]

In [41]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    # 메모리에 저장하기
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls":[
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function":{
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        }
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )
        # 실제 실행
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            # 잠깐 주석 처리
            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            # 얻어온 데이터를 다시 스트링으로 변경해야 요청할 수 있다
            if isinstance(result, str):
                tool_content = result
            else:
                tool_content = json.dumps(result, ensure_ascii=False)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": tool_content,
            })
        # 응답 메모리에 저장하기
        call_ai();
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    # AI의 응답을 통해 도구를 사용한다면 이 곳에서 처리
    process_ai_response(response.choices[0].message)
    
    

In [42]:
while True:
    message = input("Send a message to the LLM")
    if message =="quit" or message == "q":
        break
    else:
        messages.append({"role":"user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 나는 SF 영화를 좋아해
AI: SF 영화를 좋아하신다니 좋네요! 어떤 SF 영화를 좋아하시나요, 아니면 추천을 원하시나요?
User: 인셉션이랑 인터스텔라는 이미 봤어
Calling function: save_user_watched_movies with {"movie_titles":["인셉션","인터스텔라"]}
추가된 시청 영화 목록: ['인셉션', '인터스텔라']
총 시청 영화 목록: ['인셉션', '인터스텔라']
Calling function: get_popular_movies with {}
AI: 다음은 당신이 좋아할 만한 SF 영화 추천 목록입니다:

1. **Project Hail Mary**
   - **개요**: 과학 선생님인 라이랜드 그레이스는 자신의 정체성과 지구를 구할 임무를 기억해가면서 우주선에서 깨어납니다. 그가 알아내야 할 것은 태양을 죽이는 신비로운 물질의 수수께끼입니다.
   - **개봉일**: 2026-03-15
   - **평점**: 8.7
   ![Project Hail Mary](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

2. **The Mandalorian and Grogu**
   - **개요**: 제국이 무너지고, 제국의 군벌들이 은하계에 흩어진 가운데, 새로운 공화국은 전설적인 만달로리안 용병과 그의 젊은 제자가 힘을 합쳐 싸웁니다.
   - **개봉일**: 2026-05-20
   - **평점**: 6.8
   ![The Mandalorian and Grogu](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1HOsiZceDMbRr1Ca.jpg)

3. **Masters of the Universe**
   - **개요**: 15년 후 성검이 prince Adam을 에테르니아로 인도하며, 그는 자신의 가족과 세계를 구하기 위해 친구들과 힘을 합쳐야 합니다.
   - **개봉일**: 2026-06-03
